In [1]:
# Our training data
emails = [
    # Phishing examples (label = 1)
    "URGENT: Your account will be suspended in 24 hours. Click here to verify",
    "Congratulations! You've won a $500 gift card. Claim your prize immediately",
    "Dear Customer, confirm your password and SSN here to secure your account",
    "Your account has been locked. Verify your identity immediately or lose access",
    "Click this link now to claim your refund before it expires",
    "We noticed suspicious activity. Login here to confirm your identity",
    "Your payment failed. Update your billing info now to avoid suspension",
    "You have a pending prize. Click here to claim before midnight",
    "Security alert: unusual login detected. Confirm your password now",
    "Act now! Your subscription will be cancelled unless you verify today",

    # Legitimate examples (label = 0)
    "Hi Rai, just confirming our meeting tomorrow at 2pm",
    "Your Amazon order has shipped and will arrive Thursday",
    "Reminder: the monthly newsletter is attached",
    "Thanks for your email, I'll get back to you by Friday",
    "The team lunch is scheduled for 1pm in the main hall",
    "Attached is the report you asked for last week",
    "Can we reschedule our call to Monday morning instead?",
    "Your invoice for this month has been generated, no action needed",
    "Happy birthday! Hope you have a great day",
    "Here are the meeting notes from today's discussion",
]

labels = [1,1,1,1,1,1,1,1,1,1, 0,0,0,0,0,0,0,0,0,0]  # 1 = phishing, 0 = legit

print("Total emails:", len(emails))
print("Total labels:", len(labels))

Total emails: 20
Total labels: 20


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# This converts text into numbers based on word importance
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(emails)

print("Shape of our data:", X.shape)
print("Some words it learned:", vectorizer.get_feature_names_out()[:10])

Shape of our data: (20, 127)
Some words it learned: ['1pm' '24' '2pm' '500' 'access' 'account' 'act' 'action' 'activity'
 'alert']


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# Split data: most for training, some held back for testing
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=42)

# Create and train the model
model = MultinomialNB()
model.fit(X_train, y_train)

print("Model trained successfully!")
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Model trained successfully!
Training samples: 14
Testing samples: 6


In [4]:
from sklearn.metrics import accuracy_score

# Test the model on unseen data
predictions = model.predict(X_test)

# Check accuracy
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)
print("Actual labels:   ", list(y_test))
print("Predicted labels:", list(predictions))


Accuracy: 1.0
Actual labels:    [1, 0, 0, 1, 1, 1]
Predicted labels: [np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1)]


In [5]:
def classify_email(email_text):
    # Convert the new email into the same numeric format
    email_vector = vectorizer.transform([email_text])

    # Predict
    prediction = model.predict(email_vector)[0]
    probability = model.predict_proba(email_vector)[0]

    result = "PHISHING" if prediction == 1 else "LEGITIMATE"
    confidence = max(probability) * 100

    print(f"Email: {email_text}")
    print(f"Classification: {result}")
    print(f"Confidence: {confidence:.1f}%")
    print("---")

# Test it live with your own made-up examples
classify_email("Click here immediately to verify your bank password")
classify_email("Hey, are we still meeting for coffee tomorrow?")

Email: Click here immediately to verify your bank password
Classification: PHISHING
Confidence: 65.3%
---
Email: Hey, are we still meeting for coffee tomorrow?
Classification: LEGITIMATE
Confidence: 73.4%
---


In [6]:
classify_email("Dear user you have won 10000 rupee in the benazir bhuto program, to claim the money send your account number and jazz cash 50rupee.")

Email: Dear user you have won 10000 rupee in the benazir bhuto program, to claim the money send your account number and jazz cash 50rupee.
Classification: LEGITIMATE
Confidence: 54.5%
---


In [7]:
# New examples with different vocabulary
new_phishing = [
    "Dear user you have won 10000 rupee, send your account number and jazz cash 50 rupee to claim",
    "Congratulations winner! Send your CNIC number and bank details to receive your prize money",
    "You have won a lottery of 5 lakh, share your ATM pin to process withdrawal",
]

new_legit = [
    "Please find attached the salary slip for this month",
    "Reminder to submit your leave application by Friday",
    "The office will remain closed tomorrow for maintenance",
]

# Combine with original data
emails_v2 = emails + new_phishing + new_legit
labels_v2 = labels + [1,1,1] + [0,0,0]

print("New total emails:", len(emails_v2))

New total emails: 26


In [8]:
# Re-vectorize with the bigger vocabulary
vectorizer_v2 = TfidfVectorizer()
X_v2 = vectorizer_v2.fit_transform(emails_v2)

# Retrain
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_v2, labels_v2, test_size=0.3, random_state=42)
model_v2 = MultinomialNB()
model_v2.fit(X_train2, y_train2)

print("Retrained on", X_train2.shape[0], "samples")
def classify_email_v2(email_text):
    email_vector = vectorizer_v2.transform([email_text])
    prediction = model_v2.predict(email_vector)[0]
    probability = model_v2.predict_proba(email_vector)[0]
    result = "PHISHING" if prediction == 1 else "LEGITIMATE"
    confidence = max(probability) * 100
    print(f"Classification: {result} | Confidence: {confidence:.1f}%")

classify_email_v2("Dear user you have won 10000 rupee in the benazir bhutto program, to claim the money send your account number and jazz cash 50rupee")

Retrained on 18 samples
Classification: PHISHING | Confidence: 70.6%
